In [ ]:
import pandas as pd
from glob import glob
import os

In [ ]:
# 1. Configuration and Paths

data_path = "../../data/raw/Dataset(raw)"
subjects_info_path = "../../data/raw/data_subjects_info.csv"

LABEL_MAP = {
    'dws': 0, 'ups': 1, 'wlk': 2, 'jog': 3, 'sit': 4, 'std': 5
}

files = sorted(glob(data_path + "/*/*.csv"))

-B1: tạo các biến chứa đường dẫn của đối tượng và thư mục cần nghiên cứu.
-B2: đổi các tên hoạt động từ dạng chuỗi sang dạng số.
=> Lý do: máy làm việc hiệu quả hơn với con số
-B3: thu thập danh sách tất cả các file dữ liệu csv và sắp xếp chúng theo thứ tự bảng chữ cái.

In [ ]:
# 2. Read and Combine all files
all_df = pd.DataFrame()
data_set = 1

for f in files:
    folder_name = os.path.basename(os.path.dirname(f))
    activity = folder_name.split("_")[0]
    trial = folder_name.split("_")[1]
    user_id = int(os.path.basename(f).replace("sub_", "").replace(".csv", ""))

    df = pd.read_csv(f)

    df["user_id"] = user_id
    df["label"] = LABEL_MAP[activity]
    df["set"] = data_set

    all_df = pd.concat([all_df, df], ignore_index=True)
    data_set += 1

-B1: tạo một DataFrame rỗng, sau đó từ mỗi file, đoạn code sẽ lấy những thông tin quan trọng từ chính tên file.
-B2: sau khi đọc nội dung từ file csv, đoạn code sẽ lấy thêm 3 cột mới(ID người tham gia, hoạt động của người tham gia và lưu số thứ tự file đang đọc) vào DataFrame hiện tại.
-B3: gộp dữ liệu mới vào bảng tổng(all_df) và đánh số thứ tự lại hàng tránh bị trùng lặp khi gộp nhiều file.

In [ ]:
# 3. Integration: Merge with Subject Information

df_subjects = pd.read_csv(subjects_info_path)
df_subjects = df_subjects.rename(columns={'code': 'user_id'})

all_df = pd.merge(all_df, df_subjects, on='user_id', how='left')

-B1: phòng trường hợp cột để tên người dùng là code, chúng ta đổi thành user_id để khớp với user_id vừa tạo trước đó.
-B2: gộp 2 bảng lại nhưng ưu tiên bảng bên trái, nếu ID trùng nhau thì sẽ ghép các thông tin tương ứng lại. Tất cả dữ liệu cảm biến trong all_df sẽ được dữ lại.

In [ ]:
# 4. Cleaning and Feature Construction

if "Unnamed: 0" in all_df.columns:
    del all_df["Unnamed: 0"]

all_df["acc_x"] = all_df["userAcceleration.x"] + all_df["gravity.x"]
all_df["acc_y"] = all_df["userAcceleration.y"] + all_df["gravity.y"]
all_df["acc_z"] = all_df["userAcceleration.z"] + all_df["gravity.z"]

all_df.rename(columns={
    'rotationRate.x': 'gyr_x',
    'rotationRate.y': 'gyr_y',
    'rotationRate.z': 'gyr_z'
}, inplace=True)

all_df.dropna(inplace=True)

all_df = all_df.drop(
    columns = ["userAcceleration.x", "userAcceleration.y", "userAcceleration.z", "gravity.x", "gravity.y", "gravity.z"]
)
all_df = all_df[
    [
        "acc_x", "acc_y", "acc_z",
        "gyr_x", "gyr_y", "gyr_z",
        "attitude.roll", "attitude.pitch", "attitude.yaw",
        "user_id", "label", "set",
        "weight", "height", "age", "gender"
    ]
]

-B1: xóa các cột chỉ mục dư thừa(Unamed: 0).
-B2: cộng giá trị userAcceleration với gravity để thu được đặc trưng acc_x, acc_y, acc_z phản ánh chuyển động thực tế của người dùng.
=> Lý do: dữ liệu thô từ điện thoại thường tách biệt giữa gia tốc người dùng tạo ra và trọng lực.
-B3: đổi tên các cột Gyroscope thành các tên ngắn gọn và dễ hiểu hơn(gyr_x, gyr_y, gyr_z).
-B4: Xóa bỏ tất cả các hàng có chứa giá trị NaN (trống), đồng thời xóa các cột thành phần userAcceleration, gravity do không còn cần thiết nữa.
=> Giúp giảm dung lượng bộ nhớ và làm gọn bộ dữ liệu.
-B5: sắp xếp lại các cột theo một trật tự logic(Gia tốc -> Gyroscope -> Tư th(Attitude) -> Thông tin định danh -> Thông tin cá nhân).

In [ ]:
# 5. Working with Datetimes (50Hz = 20ms)

all_df["time_ms"] = all_df.groupby("set").cumcount() * 20
all_df.index = pd.to_datetime(all_df["time_ms"], unit="ms")

-B1: cho rằng cứ 20ms(50 Hz) thì một dữ liệu mới sẽ được ghi lại và nhóm dữ liệu theo từng set.
-B2: dùng cumcount() để nhân với khoảng cách thời gian(20ms) để đổi ra thời gian tương ứng tính bằng mili giây.
-B3: chuyển đổi sang DatetimeIndex(thời gian chuẩn của Pandas) để có thể sử dụng hàm như resample, sau đó gán thời gian này làm index cho toàn bộ dữ liệu.